## Web Scraping - Dashen Bank

In [24]:
# The unique identifier for Dashen Bank's app on the Google Play Store (assumed)
DASHEN_ID = 'com.dashen.dashensuperapp'

# Step 1: Get app metadata (rating, installs, description...)
dashen_app_info = app(
    DASHEN_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Dashen Bank App Info")
print("=" * 50)
print(f"App Title   : {dashen_app_info['title']}")
print(f"Current Score: {dashen_app_info['score']}")
print(f"Total Ratings: {dashen_app_info['ratings']:,}")
print(f"Total Reviews: {dashen_app_info['reviews']:,}")
print(f"Installs     : {dashen_app_info['installs']}")

Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.22913
Total Ratings: 5,618
Total Reviews: 1,023
Installs     : 1,000,000+


In [25]:

print(f"Scraping reviews for Dashen Bank...")

dashen_result, dashen_continuation_token = reviews(
    DASHEN_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,
    count=400,
    filter_score_with=None

print(f"Collected {len(dashen_result)} raw reviews for Dashen Bank")

Scraping reviews for Dashen Bank...
Collected 400 raw reviews for Dashen Bank


In [26]:
# Step 3: Extract required fields from reviews for Dashen Bank
dashen_reviews_data = []

for review in dashen_result:
    dashen_reviews_data.append({
        'review_id': review.get('reviewId', ''),
        'review_text': review.get('reviewText', review.get('content', '')),
        'rating': review.get('score', 0),
        'review_date': review.get('at', ''),
        'bank': 'Dashen Bank',
        'app_name': 'DASHEN BANK',
        'source': 'Google Play'
    })

dashen_reviews = pd.DataFrame(dashen_reviews_data)
print(f"\nRaw reviews before deduplication for Dashen Bank: {len(dashen_reviews)}")
dashen_reviews = dashen_reviews.drop_duplicates(subset=['review_id'], keep='first').reset_index(drop=True)
print(f"Reviews after deduplication by review_id for Dashen Bank: {len(dashen_reviews)}")

print("\n" + "=" * 50)
print("Dashen Bank Reviews DataFrame")
print("=" * 50)
print(f"Total reviews collected: {len(dashen_reviews)}")
print(f"\nColumns: {list(dashen_reviews.columns)}")
print(f"\nFirst 3 reviews:")
print(dashen_reviews[['bank', 'rating', 'review_date', 'source']].head(3))
print(f"\nRating distribution:")
print(dashen_reviews['rating'].value_counts().sort_index())


Raw reviews before deduplication for Dashen Bank: 400
Reviews after deduplication by review_id for Dashen Bank: 400

Dashen Bank Reviews DataFrame
Total reviews collected: 400

Columns: ['review_id', 'review_text', 'rating', 'review_date', 'bank', 'app_name', 'source']

First 3 reviews:
          bank  rating         review_date       source
0  Dashen Bank       1 2026-05-13 18:37:46  Google Play
1  Dashen Bank       1 2026-05-13 16:14:11  Google Play
2  Dashen Bank       5 2026-05-13 14:44:57  Google Play

Rating distribution:
rating
1     78
2     13
3     19
4     27
5    263
Name: count, dtype: int64


## Preprocessing - Dashen Bank

In [27]:
# View first 10 rows of the required fields only for Dashen Bank
required_columns = ['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

# Reorder the DataFrame to ensure the columns appear in the requested order
dashen_reviews = dashen_reviews[required_columns]

print("Required columns:")
print(dashen_reviews.columns.tolist())
print("\nFirst 10 reviews with the required fields:")
print(dashen_reviews.head(10))

print("\nReview text availability:")
print(f"Non-empty review_text rows: {dashen_reviews['review_text'].astype(bool).sum()}")
print(f"Empty review_text rows: {len(dashen_reviews) - dashen_reviews['review_text'].astype(bool).sum()}")

Required columns:
['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

First 10 reviews with the required fields:
                                         review_text  rating  \
0  ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ...       1   
1                          bad mobile banking at all       1   
2                                     very nice app.       5   
3                                 very difficult app       1   
4  Good app, but debit transactions not allowed W...       5   
5                                               nice       5   
6  Very bad customer service line. they won't pic...       1   
7                                          smart app       5   
8           The application booting time is so bad..       3   
9                                               good       5   

          review_date         bank     app_name       source  \
0 2026-05-13 18:37:46  Dashen Bank  DASHEN BANK  Google Play   
1 2026-05-13 16:14:11

In [28]:
# Handle missing values for Dashen Bank
dashen_reviews = dashen_reviews.dropna(subset=['review_text', 'rating'])
print(f"Total reviews after dropping missing values for Dashen Bank: {len(dashen_reviews)}")
print(f"Final dataset shape for Dashen Bank: {dashen_reviews.shape}")

Total reviews after dropping missing values for Dashen Bank: 400
Final dataset shape for Dashen Bank: (400, 7)


In [29]:
# Normalize dates to YYYY-MM-DD format for Dashen Bank
dashen_reviews['review_date'] = pd.to_datetime(dashen_reviews['review_date'], errors='coerce').dt.strftime('%Y-%m-%d')
print(f"Final dataset shape for Dashen Bank: {dashen_reviews.shape}")

Final dataset shape for Dashen Bank: (400, 7)


In [33]:
# Get the date range for Dashen Bank
dashen_start_date = dashen_reviews['review_date'].min()
dashen_end_date = dashen_reviews['review_date'].max()

print(f"Dashen Bank data spans from: {dashen_start_date}")
print(f"Up to: {dashen_end_date}")

Dashen Bank data spans from: 2025-09-28
Up to: 2026-05-13


## Data Quality Summary for Cleaned Reviews

### CBE Bank Reviews (`cbe_reviews_cleaned.csv`)

*   **Total Reviews:** The dataset contains `450` reviews.
*   **Columns:** 7 columns: `review_text`, `rating`, `review_date`, `bank`, `app_name`, `source`, `review_id`.
*   **Missing Values:** All missing `review_text` and `rating` values were handled by dropping corresponding rows. The final dataset has no missing values in these critical columns.
*   **Date Range:** Reviews span from `2026-03-10` to `2026-05-14`.
*   **Duplicates:** Duplicate reviews based on `review_id` were removed (though none were found in the initial scrape).
*   **Review Text Availability:** All `450` reviews have non-empty `review_text`.

### Dashen Bank Reviews (`dashen_reviews_cleaned.csv`)

*   **Total Reviews:** The dataset contains `400` reviews.
*   **Columns:** 7 columns: `review_text`, `rating`, `review_date`, `bank`, `app_name`, `source`, `review_id`.
*   **Missing Values:** All missing `review_text` and `rating` values were handled by dropping corresponding rows. The final dataset has no missing values in these critical columns.
*   **Date Range:** Reviews span from `2025-09-28` to `2026-05-13`.
*   **Duplicates:** Duplicate reviews based on `review_id` were removed (though none were found in the initial scrape).
*   **Review Text Availability:** All `400` reviews have non-empty `review_text`.

Both datasets are now cleaned and structured with consistent column names and formats, ready for further analysis.

In [31]:
# Save the cleaned Dashen Bank dataset as a CSV
dashen_reviews.to_csv('dashen_reviews_cleaned.csv', index=False)

In [32]:
from google.colab import files

files.download('dashen_reviews_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>